# 환경설정


In [ ]:
%pip install -q langchain langchain-openai python-dotenv


In [ ]:
# 현재 환경 확인
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


In [ ]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}

print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

# llm은 미리 생성한 LangChain 채팅 모델
PERSONA_LLMS = {
    "분석가": llm.bind(temperature=0),
    "마케터": llm.bind(temperature=0.6),
    "개발자": llm.bind(temperature=0.3),
    "기획자": llm.bind(temperature=0.7),
    "디자이너": llm.bind(temperature=0.9),
}

print(f"✅ {len(PERSONA_LLMS)}개 temperature 등록 완료: {list(PERSONA_LLMS.keys())}")

✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']
✅ 5개 temperature 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


In [ ]:
# 페르소사 소개 사전
PERSONA_INTROS = {
    "분석가": "수치와 근거 기반 답변",
    "마케터": "비즈니스 관점과 마케팅 사례 중심 답변",
    "개발자": "코드 예시를 활용한 실용적인 답변",
    "기획자": "사용자 가치와 우선순위 중심 답변",
    "디자이너": "사용자 경험과 시각적 비유로 쉬운 답변",
}

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime
from getpass import getpass
from dotenv import load_dotenv
# langchain 패키지 가져오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 현재 작업 폴더 기준
BASE_DIR = Path.cwd()
print("현재 작업 폴더:", BASE_DIR)
print("✅ import 완료")


In [ ]:
# OPENAI API 키 확인
load_dotenv(BASE_DIR / ".env")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ").strip()
print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


### 실습과제 3번


In [ ]:
# 페르소나 사전: 이름 → system 메시지
with open(BASE_DIR / "personas.json", encoding="utf-8-sig") as f:
    PERSONAS = json.load(f)

if not isinstance(PERSONAS, dict) or not PERSONAS:
    raise ValueError("personas.json에는 이름과 역할 설명이 있어야 합니다.")

print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS)}")


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "{role}"),
    ("human", "{question}"),
])

# llm은 미리 생성한 LangChain 채팅 모델
PERSONA_LLMS = {}

for name in PERSONAS:
    PERSONA_LLMS[name] = llm


## 실행


In [ ]:
while True:
    print("선택 가능:", ", ".join(PERSONAS))
    persona = input("페르소나 선택 (종료: 종료): ").strip()

    if persona in ("종료", "/quit", "/exit"):
        break

    if persona not in PERSONAS:
        print("목록에 있는 페르소나를 선택하세요.")
        continue

    user_input = input("질문: ").strip()

    if not user_input:
        continue

    persona_llm = PERSONA_LLMS[persona]
    chain = prompt | persona_llm | StrOutputParser()

    answer = chain.invoke({
        "role": PERSONAS[persona],
        "question": user_input,
    })

    print(answer)
